In [1]:
import ollama
import requests
import json

In [2]:
model = "llama3.2:1b"

#### Task 1: Interact with deployed LLM via python 


**Objective:**

Explore different techniques to interact with the deployed LLM.

**Task Description:**

1. Use Request libaray (HTTP Client) and send a POST request to interact with the LLM: [How To](https://requests.readthedocs.io/en/latest/user/quickstart/#make-a-request)

In [3]:
# Simple HTTP Request via requests

# Define the URL of the deployed LLM ( this port is forwarded from the docker container to the host system)
url = "http://localhost:11434/api/generate"

# Define the prompt as json
body_json = {
    "model": model,
    "prompt": "Describe Generative AI in two sentences."
}

# ADD HERE YOUR CODE
# HINT: Send the POST request using the json body
response = requests.post(url, json=body_json)

# Check if the request was successful
if response.status_code == 200:
    # Process the response
    response_text = response.text

    # Convert each line to json
    response_lines = response_text.splitlines()
    response_json = [json.loads(line) for line in response_lines]
    for line in response_json:
        # Print the response. No line break
        print(line["response"], end="")
else:
    print("Error:", response.status_code)


Generative AI refers to a class of artificial intelligence algorithms that can create original content, such as images, music, text, and videos, by generating new data based on patterns and associations learned from existing data. This technology enables machines to produce novel and often realistic outputs that are indistinguishable from those created by humans, raising important questions about authorship, ownership, and the potential impact on society.

**Task Description:**

2. Use Ollama python library to interact with the LLM: [How To](https://pypi.org/project/ollama/)

- First use method ``ollama.chat(...)``
- First use method ``ollama.chat(...)`` with ``stream=True``

In [4]:
# API Call via ollama

# ADD HERE YOUR CODE

client = ollama.Client(host="http://localhost:11434")

response = client.chat(
    model=model,
    messages=[
        {"role": "user", "content": "Describe Generative AI in two sentences."}
    ]
)

print(response["message"]["content"])

Generative AI is a subset of artificial intelligence that enables machines to create new content, such as text, images, and music, based on patterns and principles learned from existing data, allowing for the generation of novel and diverse outputs. Through algorithms and neural networks, generative AI systems can produce original content that is often indistinguishable from that created by humans, raising questions about authorship, creativity, and the potential impact on various industries.


In [5]:
# Streaming API Call via ollama

# Response streaming can be enabled by setting stream=True, 
# modifying function calls to return a Python generator where each part is an object in the stream.

# ADD HERE YOUR CODE

stream = client.chat(
    model=model,
    messages=[
        {"role": "user", "content": "Describe Generative AI in two sentences."}
    ],
    stream=True
)

for chunk in stream:
    print(chunk["message"]["content"], end="", flush=True)

Gener

ative

 AI

 refers

 to

 a

... (85 outputs left out)

 and

 language

 translation

.

#### Task 2: Experimenting with Prompt Techniques

**Objective:**

Objective: Explore different prompt techniques (Zero Shot, One Shot, and Few Shot) by sending different types of prompts to the LLM.

![image](https://miro.medium.com/v2/resize:fit:1400/format:webp/1*QSpK--jqPiUU_OHuZvtUWA.png)

**Task Description:**

1. Create three prompts for a sentiment analysis task: a Zero Shot prompt, a One Shot prompt, and a Few Shot prompt. Use the examples from the table above.
2. Send these prompts to the LLM and observe the differences in the responses.
3. Compare and discuss the responses.

In [6]:
# ADD HERE YOUR PROMPTS

zero_shot_prompt = """
Classify this review: I loved this movie!
Sentiment:
"""

one_shot_prompt = """
Classify this review: I loved this movie!
Sentiment: Positive

Classify this review: I don't like this chair.
Sentiment:
"""

few_shot_prompt = """
Classify this review: I loved this movie!
Sentiment: Positive

Classify this review: I don't like this chair.
Sentiment: Negative

Classify this review: Who would use this product?
Sentiment:
"""

client = ollama.Client(host="http://localhost:11434")

for name, prompt in [
    ("Zero-Shot", zero_shot_prompt),
    ("One-Shot", one_shot_prompt),
    ("Few-Shot", few_shot_prompt),
]:
    print(f"\n--- {name} ---")
    print(f"Anfrage:\n{prompt}")
    print("Antwort:")

    stream = client.chat(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        stream=True,
    )

    for chunk in stream:
        print(chunk["message"]["content"], end="", flush=True)

    print("\n")



--- Zero-Shot ---
Anfrage:

Classify this review: I loved this movie!
Sentiment:

Antwort:


The

 sentiment

 of

 this

 review

... (101 outputs left out)

-

Em

otional

".

Bei Zero-Shot erkennt das Modell die positive Bewertung ohne Beispiele und antwortet mit einem ganzen Satz.

Bei One-Shot erkennt es die neue Bewertung als negativ. Es wiederholt aber auch die Einordnung des vorgegebenen Beispiels.

Bei Few-Shot ordnet es die letzte Frage als neutral oder ambivalent ein. Diese Frage könnte auch abwertend gemeint sein. Wieder werden die Beispiele zusätzlich eingeordnet.

Mehr Beispiele führen in diesem Versuch nicht eindeutig zu besseren Antworten. Für einen fairen Vergleich müsste in allen drei Varianten dieselbe Bewertung geprüft werden.


#### Task 3: Prompt Refinement and Optimization

**Objective:** 

Refine a prompt to improve the clarity and quality of the LLM's response.

**Task Description:**

- Start with a basic prompt asking the LLM to summarize a paragraph.
- Refine the prompt by adding specific instructions to improve the summary's quality. (Example: define how long the summary should be, define on which to focus in the summary)

In [7]:
# Original prompt
original_prompt = "Summarize the following paragraph: Generative AI is a field of artificial intelligence focused on creating new content based on patterns learned from existing data. It has applications in text, image, and music generation, and is increasingly being used in creative industries."

# ADD HERE YOUR PROMPT
refined_prompt = (
    original_prompt
    + "\n\nWrite exactly two short sentences in simple English. "
    "Focus on what generative AI does and its applications. "
    "Use only information from the paragraph. "
    "Output only the summary."
)

# Stream the responses and print them
for idx, prompt in enumerate([original_prompt, refined_prompt]):
    prompt_type = ["Original Prompt", "Refined Prompt"][idx]
    print(f"\n--- {prompt_type} ---\n")
    print(f"User Prompt:\n{prompt}\n")
    
    stream = client.chat(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        stream=True,
    )
    
    print("Model Output:")
    for chunk in stream:
        print(chunk["message"]["content"], end="", flush=True)
    print("\n-----------------------------\n")



--- Original Prompt ---

User Prompt:
Summarize the following paragraph: Generative AI is a field of artificial intelligence focused on creating new content based on patterns learned from existing data. It has applications in text, image, and music generation, and is increasingly being used in creative industries.

Model Output:


Gener

ative

 AI

 is

 a

... (53 outputs left out)

 creative

 industries

.


-----------------------------



#### [Optional] Task 4: Structured Prompting with Roles (Pirate Theme)

**Objective:**

Learn how to use structured prompts that combine role assignment, clear instructions, and examples to improve the output of language models. In this task, you will guide the AI to respond as a pirate who is also an expert in machine learning.

**Instructions:**

- Role Assignment: In your prompt, specify the role of the AI as a Machine Learning Expert who speaks like a pirate.

- Instruction: Clearly state what you want the AI to explain or discuss in pirate language.

- Examples: Provide examples to guide the AI in using pirate lingo while explaining technical concepts.

In [8]:
# Combined Techniques Prompt with Pirate Theme

structured_prompt = """
Role:
You are a machine learning expert who speaks like a pirate.

Task:
Explain supervised learning to a beginner in 3 short sentences.
Use pirate language, but keep the explanation technically correct.

Examples of the style:
- Arrr, training data be examples a model learns from!
- A label be the correct answer attached to a training example, matey!

Now explain supervised learning using a spam filter as an example.
"""

client = ollama.Client(host="http://localhost:11434")

print("=== User Prompt ===")
print(structured_prompt)

stream = client.chat(
    model=model,
    messages=[{"role": "user", "content": structured_prompt}],
    stream=True,
)

print("\n=== Model Output ===")
for chunk in stream:
    print(chunk["message"]["content"], end="", flush=True)

print("\n")


=== User Prompt ===

Role:
You are a machine learning expert who speaks like a pirate.

Task:
Explain supervised learning to a beginner in 3 short sentences.
Use pirate language, but keep the explanation technically correct.

Examples of the style:
- Arrr, training data be examples a model learns from!
- A label be the correct answer attached to a training example, matey!

Now explain supervised learning using a spam filter as an example.


=== Model Output ===


I

 be

 happy

 to

 explain

... (206 outputs left out)

em

 out

 correctly

.

In [9]:
print()
print("Fertig")



Fertig
